# 02b - Client's web Keyword Scraping (BeautifulSoup)

Extraction and cleaning of commercial tourist destinations from "www.tempsdoci.com" using Requests and BeautifulSoup. This process gathers client's data to establish target keywords for the Google Ads Planner.

### Legal & Ethical Notice

This notebook scrapes publicly accessible web pages to extract 
destination keyword data for academic research purposes.

**Scope:** Only public destination listing pages are accessed. No authentication, personal data, or paywalled content is involved.

**robots.txt:** Verified on 2026-06-03. The scraped URL (`/agenda`) is explicitly permitted (`User-agent: * / Allow: /`).

**Purpose:** Academic final project and portfolio. 

## 1. Imports

In [6]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from unidecode import unidecode
import pandas as pd

## 2. Paths configuration

In [7]:
INTERIM_PATH_RAW = Path("../data/interim/raw")

INTERIM_PATH_RAW.mkdir(
    parents=True,
    exist_ok=True
)

## 3. Scraping Configuration

### Exclusion Criteria
These terms were identified during manual review of the raw scraping output.
They are excluded because the scraper captured them as false positives:
navigation links, legal pages, product categories, contact info and audience 
segments that appear as anchor tags (`<a>`) in the website's HTML but do not 
represent searchable geographic destinations.

Exclusion was applied iteratively after inspecting the raw Excel export.

In [8]:
URL = "https://tempsdoci.com/agenda"

HEADERS = {"User-Agent": "Mozilla/5.0"}

EXCLUDED = {
    "contacto",
    "newsletter",
    "politica",
    "cookies",
    "facebook",
    "instagram",
    "linkedin",
    "inicio",
    "blog",
    "aviso legal",
    "suscribete",
    "whatsapp",
    "madrid",
    "barcelona",
    "34 627 90 23 52",
    "91 737 05 77",
    "93 323 34 23",
    "africa",
    "america",
    "asia",
    "buscador de viajes",
    "calle provenza, 212, 08036 barcelona",
    "calle velazquez, 57, bajo int. 28001 madrid",
    "cerrar",
    "circuitos",
    "compromiso",
    "con playa",
    "condiciones de contratacion",
    "culturales",
    "de naturaleza",
    "declaracion de accesibilidad de la web",
    "europa",
    "fin de ano",
    "fuera de temporada",
    "informacion legal",
    "otono",
    "por tipo",
    "preguntas frecuentes",
    "proximas salidas",
    "proyectos",
    "puente de diciembre",
    "quienes somos",
    "reviews",
    "safari",
    "salidas regulares",
    "temps d'oci",
    "temps d'oci incoming spain",
    "temps d'oci responsable",
    "terminos de servicio",
    "termsfeed generator",
    "todos los viajes",
    "tours",
    "turismo responsable",
    "verano",
    "viajes a medida",
    "viajes en grupo"    
}

## 4. Core Scraping Methods

In [9]:
def normalize_text(text):
    """
    Standardizes text by converting to lowercase, removing accents, and stripping whitespace.
    Args: text (str): The raw text string to normalize.
    Returns: str or None: The cleaned string, or None if the input is missing (NaN).
    """
    if pd.isna(text):
        return None

    text = str(text).lower().strip()
    text = unidecode(text)
    text = text.replace("-", " ")

    return text

In [10]:
def get_html(url):
    """
    Fetches the HTML content of a given URL using a GET request.
    
    Implements robust error handling by raising an exception for bad HTTP 
    status codes (e.g., 404, 500) to prevent parsing invalid pages.

    Args: url (str): The target website URL.

    Returns: str: The raw HTML string.
    """
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    print(f"✔ HTTP Status: {response.status_code} (Success)")
    return response.text

In [11]:
def create_soup(html):
    """
    Parses raw HTML text into a navigable BeautifulSoup object.
    Args: html (str): The raw HTML string.
    Returns: BeautifulSoup: The parsed HTML tree.
    """
    soup = BeautifulSoup(html, "html.parser")
    return soup

In [12]:
def extract_tours(soup):
    """
    Locates and extracts all HTML container elements representing individual tours.
    Args: soup (BeautifulSoup): The parsed HTML tree.
    Returns: list: A list of BeautifulSoup tag objects containing tour data.
    """
    tours = soup.find_all("div", class_="tour_data")
    return tours

In [13]:

def process_tours(tours):
    """
    Iterates through tour elements, extracting titles and URLs, and filters them.

    Args: tours (list): A list of BeautifulSoup tag objects.

    Returns: list: A list of dictionaries containing the processed destination data.
    """
    results = []
    for tour in tours:
        link = tour.find("a")
        if not link:
            continue
        title = link.get_text(strip=True)
        href = link.get("href")
        if not title or not href:
            continue
        search_term = create_search_term(title)
        results.append({
            "raw_title": title,
            "search_term": search_term,
            "parent_country": None,
            "keyword_type": None,
            "source": "tempsdoci",
            "url": href
        })

    return results

In [14]:
def create_search_term(text):
    """
    Cleans and truncates a raw tour title to create a concise search term.
    
    Splits the string at specific delimiters (':' or '|') to remove marketing 
    fluff, keeping only the primary destination name.

    Args: text (str): The raw tour title.
    Returns: str or None: The isolated and normalized geographic search term.
    """
    if pd.isna(text):
        return None
    
    text = str(text)
    separators = [":","|"]
    for sep in separators:
        if sep in text:
            text = text.split(sep)[0]

    return (
        normalize_text(text)
        .strip()
    )

In [15]:
def build_dataframe(results):
    """
    Converts the processed tour dictionaries into a standardized pandas DataFrame.

    Args: results (list): The list of parsed tour dictionaries.

    Returns: pd.DataFrame: A deduplicated and sorted DataFrame ready for export.
    """
    df = (
        pd.DataFrame(results)
        .drop_duplicates()
        .sort_values("search_term")
        .reset_index(drop=True)
    )

    return df

## 5. Scraper Orchestration

In [16]:
def main_scraping():
    """
    Orchestrates the complete static web scraping workflow.
    
    Requests the HTML, parses it, extracts tour containers, processes the data,
    and returns a clean DataFrame.

    Returns: pd.DataFrame: The final compiled dataset of destination keywords.
    """
    html = get_html(URL)
    soup = create_soup(html)
    tours = extract_tours(soup)
    results = process_tours(tours)
    df = build_dataframe(results)

    return df

## 6. Job Execution

In [17]:
df_tempsdoci = main_scraping()
df_tempsdoci.head()

✔ HTTP Status: 200 (Success)


,raw_title,search_term,parent_country,keyword_type,source,url
0,"ALBANIA, MACEDONIA DEL NORTE Y CORFÚ: Balcanes...","albania, macedonia del norte y corfu",None,None,tempsdoci,/viajes/viaje-en-grupo-albania-macedonia-corfu...
1,"AZERBAIYAN, GEORGIA Y ARMENIA: Cáucaso, histor...","azerbaiyan, georgia y armenia",None,None,tempsdoci,/viajes/viaje-en-grupo-azerbaiyan-georgia-arme...
2,"BOLIVIA: Ciudades coloniales, desiertos blanc...",bolivia,None,None,tempsdoci,/viajes/viaje-en-grupo-bolivia-verano
3,BUCAREST a TRANSILVANIA: Entre Mercadillos Nav...,bucarest a transilvania,None,None,tempsdoci,/viajes/viaje-en-grupo-rumania-transilvania-pu...
4,"BULGARIA: Historia, monasterios de Rila y Mar ...",bulgaria,None,None,tempsdoci,/viajes/viaje-en-grupo-bulgaria-verano


## 7. Data Validation

In [18]:
print(f"Dataset Shape: {df_tempsdoci.shape}\n")

print("--- NULL & EMPTY VALUES ---")
print(df_tempsdoci.isnull().sum())
print(f"\nEmpty 'search_term' strings: {(df_tempsdoci['search_term'] == '').sum()}")

print("\n--- DUPLICATES ---")
print(f"Duplicate URLs: {df_tempsdoci.duplicated(subset=['url']).sum()}")
print(f"Duplicate Search Terms: {df_tempsdoci.duplicated(subset=['search_term']).sum()}")

print("\n--- SUMMARY ---")
print(f"Total unique keywords extracted: {df_tempsdoci['search_term'].nunique()}")

Dataset Shape: (66, 6)

--- NULL & EMPTY VALUES ---
raw_title          0
search_term        0
parent_country    66
keyword_type      66
source             0
url                0
dtype: int64

Empty 'search_term' strings: 0

--- DUPLICATES ---
Duplicate URLs: 0
Duplicate Search Terms: 10

--- SUMMARY ---
Total unique keywords extracted: 56


## 8. Export dataset

In [ ]:
df_tempsdoci.to_parquet(
    INTERIM_PATH_RAW / "02b_tempsdoci_scraped_raw.parquet"
)

df_tempsdoci.to_excel(
    INTERIM_PATH_RAW / "02b_tempsdoci_scraped_raw.xlsx",
    index=False
)

## 9. Manual Enrichment Guidelines

The exported Excel file requires manual data enrichment before the next stage. 
It is necessary to review the file and perform the following tasks:

* Remove any remaining irrelevant commercial terms.
* Standardize commercial names if necessary.
* Assign the correct `parent_country`.
* Define the `keyword_type` for each entry (country / region / destination brand).
* Validate commercial tourist keywords.

Save the fully reviewed file in: 

`../data/interim/reviewed/azulmarino_destinations_reviewed.xlsx` (notebook 2a)  
`../data/interim/reviewed/tempsdoci_destinations_reviewed.xlsx` 

## Scraping Configuration

### Filtering Strategy
Unlike Notebook 02a (AzulMarino), this scraper does **not require an 
exclusion list** at the code level.

TempsDoci's HTML structure uses a dedicated container (`div.tour_data`) 
that only wraps actual tour listings. This structural filter eliminates 
navigation links, legal pages and contact info at the extraction level,
making a keyword-based exclusion list redundant.

Residual false positives (if any) are handled during the **manual review 
stage** documented at the end of this notebook.